## Importing Packages

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.impute import SimpleImputer

pd.set_option('display.max_rows',100)

## Import Feature Engineered Dataset

In [2]:
df = pd.read_pickle('../data/lending_club_feature_engineered.pkl')

print(df.shape)
df.head()

(1340973, 128)


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,repayment_capacity_score,creditworthiness_score,financial_stability_Score,loan_to_income_ratio_capped,funded_to_income_ratio_capped,revol_bal_to_income_ratio_capped,installment_to_income_ratio_capped,open_acc_to_total_acc_ratio_capped,debt_burden_score_capped,financial_stability_score
100,30000,30000,30000.0,36 months,22.35,1151.16,D,D5,Supervisor,5 years,...,-0.860379,3.011861,-0.846094,0.300000,0.300000,0.156030,0.138139,0.578947,67.46,-0.846094
152,40000,40000,40000.0,60 months,16.14,975.71,C,C4,Assistant to the Treasurer (Payroll),< 1 year,...,-1.759874,2.087647,NaN,0.500000,0.500000,0.777133,0.200983,0.486486,115.03,NaN
170,20000,20000,20000.0,36 months,7.56,622.68,A,A3,Teacher,10+ years,...,-0.220578,0.975468,0.706151,0.200000,0.200000,0.254160,0.074722,0.473684,48.82,0.706151
186,4500,4500,4500.0,36 months,11.31,147.99,B,B3,Accounts Examiner III,10+ years,...,-0.322414,1.441015,1.181911,0.116883,0.116883,0.116156,0.046127,0.480000,19.94,1.181911
215,8425,8425,8425.0,36 months,27.27,345.18,E,E5,Senior Director Risk Management,3 years,...,0.986400,2.714255,-0.018850,0.025688,0.025667,0.081804,0.010252,0.567568,78.07,-0.018850


## Target Variable

In [3]:
target = df['default_flag']

df['default_flag'].value_counts()


default_flag
0    1043940
1     297033
Name: count, dtype: int64

In [4]:
df['default_flag'].value_counts(normalize=True)

default_flag
0    0.778494
1    0.221506
Name: proportion, dtype: float64

## Remove Obvious Leakage Columns 

In [5]:
leakage_cols = [

    "out_prncp",
    "out_prncp_inv",

    "total_pymnt",
    "total_pymnt_inv",

    "total_rec_prncp",
    "total_rec_int",

    "recoveries",
    "collection_recovery_fee",

    "last_pymnt_amnt"
]

available_leakage_cols =  [cols for cols in leakage_cols if cols in df.columns]
df = df.drop(columns=available_leakage_cols)
print(df.shape)

(1340973, 119)


## Dependent & Independent Variables

In [6]:
X = df.drop(columns='default_flag')
y = df['default_flag']

print(f'Shape of X {X.shape}')
print(f'Shape of y {y.shape}')

Shape of X (1340973, 118)
Shape of y (1340973,)


## Feature Types

In [7]:
cat_cols = X.select_dtypes(include="object").columns.to_list()
num_cols = X.select_dtypes(include=["int64","int32","float64"]).columns.to_list()

print(f'Categorical Column Counts = {len(cat_cols)}')
print(f'Numerical Column Counts = {len(num_cols)}')

Categorical Column Counts = 14
Numerical Column Counts = 100


## Train Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.80,stratify=y,random_state=42)
print(X_train.shape)
print(X_test.shape)

(1072778, 118)
(268195, 118)


## Handling Missing Values

### Numerical Variables

In [9]:
num_impute = SimpleImputer(strategy='median')
X_train_num = pd.DataFrame(num_impute.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
X_test_num = pd.DataFrame(num_impute.fit_transform(X_test[num_cols]), columns=num_cols, index=X_test.index)

### Categorical Variables

In [10]:
cat_impute = SimpleImputer(strategy='most_frequent')
X_train_cat = pd.DataFrame(cat_impute.fit_transform(X_train[cat_cols]),columns=cat_cols, index=X_train.index)
X_test_cat = pd.DataFrame(cat_impute.fit_transform(X_test[cat_cols]), columns=cat_cols, index=X_test.index)

## One Hot Encoding

In [ ]:
X_train_cat = pd.get_dummies(X_train_cat, drop_first=True)
X_test_cat = pd.get_dummies(X_test_cat, drop_first=True)